# 01 - Build the Listeria feasibility manifest

This notebook creates the first inspectable dataset artifact for the project. It downloads the public supplementary workbook associated with Gmeiner et al. (2025), identifies the isolate-level metadata table, applies the published benzalkonium-chloride (BC) threshold, and samples a reproducible feasibility cohort.

Published endpoint:

```text
BC MIC >= 1.25 mg/L -> tolerant
BC MIC <  1.25 mg/L -> sensitive
```

Reference: https://doi.org/10.1038/s41598-025-94321-6

## 1. Configuration

The default cohort contains 200 isolates. Change `TARGET_SAMPLE_SIZE` to `100` for a faster first pass. The sampling strategy retains the observed tolerant/sensitive ratio while using a fixed random seed.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
from urllib.error import URLError
from urllib.request import urlretrieve

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTICLE_PMC_ID = "PMC11947258"
SUPPLEMENTARY_WORKBOOK_URL = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41598-025-94321-6/MediaObjects/41598_2025_94321_MOESM1_ESM.xlsx"
BC_THRESHOLD_MG_L = 1.25
TARGET_SAMPLE_SIZE = 200
RANDOM_SEED = 42

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "gmeiner_2025"
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"
RAW_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## 2. Download Supplementary Material 1

The publisher provides Supplementary Material 1 as an XLSX workbook. If the automated download stops working, download Supplementary Material 1 manually from the [article page](https://www.nature.com/articles/s41598-025-94321-6) and place the `.xlsx` file in `data/raw/gmeiner_2025/`.

In [ ]:
workbook_download_path = RAW_DIR / "41598_2025_94321_MOESM1_ESM.xlsx"
if not workbook_download_path.exists():
    try:
        print("Downloading:", SUPPLEMENTARY_WORKBOOK_URL)
        urlretrieve(SUPPLEMENTARY_WORKBOOK_URL, workbook_download_path)
    except URLError as error:
        print("Automatic download failed:", error)
        print("Download Supplementary Material 1 manually from the article page and place it in:", RAW_DIR)
else:
    print("Using cached workbook:", workbook_download_path)

xlsx_files = sorted(RAW_DIR.glob("*.xlsx"))
print("Workbook candidates:")
for path in xlsx_files:
    print(" -", path.relative_to(PROJECT_ROOT))

## 3. Inspect workbook sheets

The supplementary workbook contains title rows above the actual column headers. This cell detects the most likely header row for each sheet, then shows the cleaned sheet names, dimensions, and column names before any selection is made.

In [ ]:
if not xlsx_files:
    raise FileNotFoundError("No .xlsx workbook found. Place Supplementary Material 1 in data/raw/gmeiner_2025/ and rerun this cell.")

workbook_path = max(xlsx_files, key=lambda path: path.stat().st_size)
print("Selected workbook:", workbook_path.relative_to(PROJECT_ROOT))

def detect_header_row(workbook: Path, sheet_name: str, max_rows: int = 12) -> int:
    preview = pd.read_excel(workbook, sheet_name=sheet_name, header=None, nrows=max_rows)
    tokens = ("accession", "isolate", "strain", "mic", "lineage", "source", "id", "model")

    def score(row: pd.Series) -> tuple[int, int]:
        values = [str(value).strip().lower() for value in row.dropna()]
        token_hits = sum(any(token in value for value in values) for token in tokens)
        return token_hits, len(values)

    return max(range(len(preview)), key=lambda row_index: score(preview.iloc[row_index]))

excel_file = pd.ExcelFile(workbook_path)
header_rows = {
    sheet_name: detect_header_row(workbook_path, sheet_name)
    for sheet_name in excel_file.sheet_names
}
sheets = {
    sheet_name: pd.read_excel(workbook_path, sheet_name=sheet_name, header=header_row)
    for sheet_name, header_row in header_rows.items()
}
for sheet_name, frame in sheets.items():
    print(f"\n[{sheet_name}] header_row={header_rows[sheet_name] + 1} rows={len(frame)} columns={len(frame.columns)}")
    print(list(frame.columns))

## 4. Select and normalize the isolate table

The helper functions search common column-name variants. Review the detected mapping before continuing. If a column is not found, update the candidate list after inspecting the previous cell.

In [ ]:
def normalize_name(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower()).strip("_")

def find_column(frame: pd.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    normalized = {normalize_name(column): column for column in frame.columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]
    for key, column in normalized.items():
        if any(normalize_name(candidate) in key for candidate in candidates):
            return column
    if required:
        raise KeyError(f"Could not find any of {candidates}. Available columns: {list(frame.columns)}")
    return None

def score_sheet(frame: pd.DataFrame) -> int:
    names = [normalize_name(column) for column in frame.columns]
    tokens = ("accession", "isolate", "strain", "benzalkonium", "bc_mic", "mic_bc")
    return sum(any(token in name for name in names) for token in tokens)

def repair_embedded_header(frame: pd.DataFrame) -> pd.DataFrame:
    current_score = score_sheet(frame)
    candidate_rows = frame.head(5)
    if candidate_rows.empty:
        return frame

    candidate_scores = candidate_rows.apply(
        lambda row: score_sheet(pd.DataFrame(columns=row.dropna().tolist())),
        axis=1,
    )
    best_index = candidate_scores.idxmax()
    if candidate_scores.loc[best_index] <= current_score:
        return frame

    repaired = frame.loc[best_index + 1:].copy()
    repaired.columns = frame.loc[best_index].tolist()
    return repaired.reset_index(drop=True)

sheets = {sheet_name: repair_embedded_header(frame) for sheet_name, frame in sheets.items()}

sheet_name, source = max(sheets.items(), key=lambda item: (score_sheet(item[1]), len(item[1])))
source = source.copy()
print("Selected sheet:", sheet_name, "rows=", len(source))

column_map = {
    "isolate_id": find_column(source, ["isolate_id", "isolate", "strain_id", "strain", "sample_id"]),
    "ena_accession": find_column(source, ["ena_accession", "accession", "run_accession", "sra_accession", "wgs_accession"]),
    "mic_bc_mg_l": find_column(source, ["mic_bc_mg_l", "bc_mic", "mic_bc", "benzalkonium_chloride_mic", "benzalkonium"]),
    "group": find_column(source, ["clonal_complex", "cc", "poppunk_cluster", "lineage", "sequence_type", "st"], required=False),
    "source_metadata": find_column(source, ["source", "isolation_source", "origin"], required=False),
}
column_map

## 5. Build the feasibility cohort

The sampling is stratified by the binary BC label. The notebook writes a candidate manifest that remains easy to inspect and edit. A lineage-aware `group` is required before generating train/dev/test splits. If the workbook does not provide one, fill it later using clonal-complex metadata or PopPUNK output.

In [ ]:
manifest = pd.DataFrame({
    "isolate_id": source[column_map["isolate_id"]].astype("string").str.strip(),
    "ena_accession": source[column_map["ena_accession"]].astype("string").str.strip(),
    "mic_bc_mg_l": pd.to_numeric(source[column_map["mic_bc_mg_l"]], errors="coerce"),
})

if column_map["group"] is not None:
    manifest["group"] = source[column_map["group"]].astype("string").fillna("").str.strip()
else:
    manifest["group"] = ""

if column_map["source_metadata"] is not None:
    manifest["source_metadata"] = source[column_map["source_metadata"]].astype("string").fillna("").str.strip()
else:
    manifest["source_metadata"] = ""

manifest = manifest.dropna(subset=["isolate_id", "ena_accession", "mic_bc_mg_l"]).drop_duplicates("isolate_id")
manifest["label"] = (manifest["mic_bc_mg_l"] >= BC_THRESHOLD_MG_L).astype(int)
manifest["assembly_path"] = manifest["isolate_id"].map(lambda value: f"data/raw/assemblies/{value}.fasta")
manifest["source"] = "Gmeiner_et_al_2025_SciRep"

print("Eligible isolates:", len(manifest))
print("Full class counts:")
print(manifest["label"].value_counts().sort_index())

sample_size = min(TARGET_SAMPLE_SIZE, len(manifest))
cohort = (
    manifest.groupby("label", group_keys=False)
    .apply(lambda frame: frame.sample(n=max(1, round(sample_size * len(frame) / len(manifest))), random_state=RANDOM_SEED))
    .drop_duplicates("isolate_id")
)

if len(cohort) > sample_size:
    cohort = cohort.sample(n=sample_size, random_state=RANDOM_SEED)
elif len(cohort) < sample_size:
    remaining = manifest[~manifest["isolate_id"].isin(cohort["isolate_id"])]
    cohort = pd.concat([cohort, remaining.sample(n=sample_size - len(cohort), random_state=RANDOM_SEED)])

cohort = cohort[["isolate_id", "assembly_path", "label", "group", "mic_bc_mg_l", "source", "ena_accession", "source_metadata"]]
cohort = cohort.sort_values(["label", "isolate_id"]).reset_index(drop=True)

print("\nFeasibility cohort:", len(cohort))
print(cohort["label"].value_counts().sort_index())
print("Missing group values:", (cohort["group"] == "").sum())
cohort.head()

## 6. Export the manifest and provenance report

In [ ]:
output_path = MANIFEST_DIR / "isolate_manifest.csv"
cohort.to_csv(output_path, index=False)

provenance = pd.DataFrame([
    {"field": "article_doi", "value": "10.1038/s41598-025-94321-6"},
    {"field": "pmc_id", "value": ARTICLE_PMC_ID},
    {"field": "workbook", "value": str(workbook_path.relative_to(PROJECT_ROOT))},
    {"field": "worksheet", "value": sheet_name},
    {"field": "bc_threshold_mg_l", "value": BC_THRESHOLD_MG_L},
    {"field": "target_sample_size", "value": TARGET_SAMPLE_SIZE},
    {"field": "actual_sample_size", "value": len(cohort)},
    {"field": "random_seed", "value": RANDOM_SEED},
])
provenance_path = MANIFEST_DIR / "isolate_manifest_provenance.csv"
provenance.to_csv(provenance_path, index=False)

print("Wrote:", output_path.relative_to(PROJECT_ROOT))
print("Wrote:", provenance_path.relative_to(PROJECT_ROOT))
if (cohort["group"] == "").any():
    print("\nAction required: populate lineage-aware group values before running scripts/make_splits.py")

## 7. Next step

Review the exported manifest. Once `group` contains lineage-aware values, download a small FASTQ subset:

```bash
python scripts/download_ena_fastq.py \
  --manifest data/manifests/isolate_manifest.csv \
  --output-dir data/raw/fastq \
  --limit 10
```

The `--limit 10` smoke test is intentional. Validate data retrieval before downloading the full feasibility cohort.